In [ ]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes -q

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import time

model_name = "Qwen/Qwen3-8B"

bnb_cfg = BitsAndBytesConfig(
    load_in_8bit=True,
)

load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_cfg,
    torch_dtype=torch.float16,
    device_map="auto",
)

load_time = time.perf_counter() - load_start

print(f"Model loaded in {load_time:.1f}s")

if torch.cuda.is_available():
    print(
        f"GPU memory allocated: "
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
    )
    print(
        f"GPU memory reserved: "
        f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
    )


def generate_text(prompt, max_new_tokens=1500, thinking=False):

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    input_tokens = inputs["input_ids"].shape[1]

    print(
        f"Input tokens: {input_tokens} | "
        f"thinking: {thinking}"
    )

    if thinking:
        gen_kwargs = dict(
            do_sample=True,
            temperature=0.6,
            top_p=0.95,
            top_k=20
        )
    else:
        gen_kwargs = dict(
            do_sample=False
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    generation_start = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    generation_time = time.perf_counter() - generation_start

    generated_tokens = outputs[0][input_tokens:]
    generated_token_count = len(generated_tokens)

    tokens_per_second = (
        generated_token_count / generation_time
        if generation_time > 0
        else 0
    )

    print(f"Generated tokens: {generated_token_count}")
    print(f"Generation time: {generation_time:.1f}s")
    print(f"Generation speed: {tokens_per_second:.2f} tokens/sec")

    decoded = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    if thinking:
        if "</think>" not in decoded:
            raise ValueError(
                "Thinking did not finish. Increase max_new_tokens."
            )

        decoded = decoded.rpartition("</think>")[2].strip()

    return decoded

In [ ]:
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    allocated = torch.cuda.memory_allocated(i)
    reserved = torch.cuda.memory_reserved(i)
    gb = 1024 ** 3

    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Total:     {total / gb:.2f} GB")
    print(f"  Used:      {(total - free) / gb:.2f} GB (everything on the GPU)")
    print(f"  Free:      {free / gb:.2f} GB")
    print(f"  Tensors:   {allocated / gb:.2f} GB (model weights + live data)")
    print(f"  Reserved:  {reserved / gb:.2f} GB (PyTorch cache)")

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
NGROK_TOKEN = secrets.get_secret("NGROK_TOKEN")
API_KEY = secrets.get_secret("API_KEY")

In [ ]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    thinking = data.get("thinking", False)
    default_tokens = 3000 if thinking else 1500
    try:
        response = generate_text(
            data.get("prompt", ""),
            data.get("max_new_tokens", default_tokens),
            thinking,
        )
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    return {"response": response}


def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)